# Introdução

: Apresentação do contexto, do modelo formal por trás da hipótese e descrição clara da própria hipótese

Contextualização da desigualdade em SP e a apresentação do **Modelo Formal**.
- **Modelo Formal:** Exemplo: $Nota = \beta_0 + \beta_1(EscolaridadePais) + \beta_2(Renda) + \epsilon$. 
- **Hipótese:** Existe uma disparidade significativa nas notas do ENEM entre diferentes zonas de SP, correlacionada com indicadores socioeconômicos locais.

# Dados

descrição da base de dados e suas principais características

carrega o CSV do ENEM e faz os primeiros filtros
- `df.info()`, `df.describe()` e `df.shape`
- Fonte
- limpeza
- estatística descritiva

Dicionário para evitar nomes errados
Retirada de colunas irrelevantes
Outliers (média no agrupamento por munícipio)

Fontes:

## Merge

In [ ]:
import pandas as pd

# 1. Carregar seus dados processados do ENEM
df_enem = pd.read_csv('data/processed/enem_sp_completo_agrupado.csv')

# 2. Carregar o diretório de municípios (para garantir os nomes e siglas corretas)
df_dicionario_sp = pd.read_csv('data/raw/br_bd_diretorios_brasil_municipio.csv.gz')
df_dicionario_sp = df_dicionario_sp[df_dicionario_sp['sigla_uf'] == 'SP'][['id_municipio', 'nome']]

# 3. Carregar os dados de IDH/Infraestrutura
df_idh = pd.read_csv('data/raw/mundo_onu_adh_municipio.csv.gz')
# Filtrar apenas o ano mais recente (2010) para não duplicar linhas
df_idh = df_idh[df_idh['ano'] == 2010]

# Primeiro: Unimos o ENEM com o diretório para ter os nomes corretos das cidades
df = df_enem.merge(df_dicionario_sp, left_on='CO_MUNICIPIO_ESC', right_on='id_municipio')

# Segundo: Unimos com os dados de IDH/Infraestrutura
df = df.merge(df_idh, on='id_municipio')

# Visualizar o resultado
print(f"Total de municípios após o merge: {len(df)}")
municipios_sp = df['nome'].unique().tolist()
municipios_sp.sort()
print(f"Municípios de SP presentes no dataset: {municipios_sp}")
print(df.columns.tolist())

## Feature engineering

In [ ]:
df['nota_final'] = (df["NU_NOTA_CN"]+df['NU_NOTA_CH']+ df['NU_NOTA_LC'] +df['NU_NOTA_MT']+ df['NU_NOTA_REDACAO'])/5
municipios = ['Ferraz de Vasconcelos', 'Indaiatuba', 'Jundiaí', 'Itaquaquecetuba', 'São Paulo', 'Poá', 'Osasco','Guarulhos','São Bernardo' ]
for i in municipios:
    df1 = df[df['nome']== i]
    colunas = ["nome",'nota_final', "idhm"]
    print(df1[colunas])
    print("---"*30)

# Média do Brasil é 546

In [ ]:

# Colunas de interesse:
colunas = ['nome', 'nota_final', 'id_municipio', 'idhm', "idhm_e", "idhm_l", "idhm_r", "indice_frequencia_escolar", "indice_escolaridade","taxa_sem_energia_eletrica", "taxa_agua_encanada", 'taxa_atividade_15_17', 'renda_media_ocupados', 'renda_pc', 'renda_pc_pobreza_extrema', 'prop_vulner_pobreza_criancas', 'prop_pobreza', 'indice_gini', 'prop_pobreza_criancas','taxa_freq_fundamental_15_17', 'taxa_analfabetismo_15_a_17']
df_final = df[colunas].set_index(['id_municipio', 'nome'])
print(df_final.head())

## Limpeza de dados nulos

In [ ]:
#
print(df_final.isna().sum())
# Qual o nome do munícipio com nota final faltando?
municipios_faltando_nota = df_final[df_final['nota_final'].isna()].index.get_level_values('nome').unique().tolist()
print(f"Municípios com nota final faltando: {municipios_faltando_nota}")
df_final.dropna(subset=['nota_final'], inplace=True)
print(df_final.isna().sum())
# Quantidade de colunas agora:
print(f"Quantidade de linhas após remover linhas com nota final faltando: {df_final.shape[0]}")

In [ ]:
# Ver se tem outliers pelo método do IQR:
Q1 = df_final['nota_final'].quantile(0.25)
Q3 = df_final['nota_final'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
outliers = df_final[(df_final['nota_final'] < limite_inferior) | (df_final['nota_final'] > limite_superior)]
print(f"Municípios considerados outliers: {outliers.index.get_level_values('nome').tolist()}")
# Notas consideradas outliers:
print(outliers['nota_final'].tolist())

In [ ]:
# Winsorização:
df_final['nota_final'] = df_final['nota_final'].clip(lower=limite_inferior, upper=limite_superior)
# Comparar as notas antes e depois da winsorização nos múnicipios considerados outliers:
df_outliers = df_final.loc[outliers.index]
print(df_outliers[['nota_final']])

## Estatística Descritiva

In [ ]:
df_final.info()

In [ ]:
df_final.describe()

In [ ]:
df_final.shape

# Resultados

## Gráficos de dispersão

## Matriz de correlação

Apresentação das evidências empíricas relacionadas à hipótese (gráficos e estatísticas)

Gráficos (quais?) e o que cada um revela sobre a hipótese inicial

In [ ]:
import pandas as pd
variavel_de_notas = 'nota_final'
# Calculamos a correlação pelos metodos de spearman e pearson para descobrirmos as variaveis que mais se correlacionam com a nota a fim de compará-las
spearman = df_final.corr(method='spearman')[variavel_de_notas].abs()
pearson = df_final.corr(method='pearson')[variavel_de_notas].abs()
# Visualização dos resultados do método spearman
print('Váriaveis que mais se relacionam com a nota pelo método Spearman')
maiores_spearman = spearman.drop(variavel_de_notas).sort_values(ascending=False).head(10)
print(maiores_spearman)
# Visualização dos resultados do método pearson
print('Váriaveis que mais se relacionam com a nota pelo método Pearson')
maiores_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=False).head(10)
print(maiores_pearson)

### Correlação positiva

### Correlação negativa

### Correlação fracas

In [ ]:
print('Váriaveis que menos se relacionam com a nota pelo método Spearman')
menores_spearman = spearman.drop(variavel_de_notas).sort_values(ascending=True).head(10)
print(menores_spearman)
# Visualização dos resultados do método pearson
print('Váriaveis que menos se relacionam com a nota pelo método Pearson')
menores_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=True).head(10)
print(menores_pearson)

## Geopandas

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Carregar o mapa de municípios de SP que baixou
mapa_sp = gpd.read_file('data/shp/SP_Municipios_2024.shp')

# Garantir que o ID do município é string para o cruzamento
mapa_sp['CD_MUN'] = mapa_sp['CD_MUN'].astype(str)
df_final_reset = df_final.reset_index()
df_final_reset['id_municipio'] = df_final_reset['id_municipio'].astype(str)

# Unir os dados alfanuméricos com o mapa geométrico
mapa_final = mapa_sp.merge(df_final_reset, left_on='CD_MUN', right_on='id_municipio')
# Plotar o mapa de calor da nota de Matemática
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
mapa_final.plot(column='nota_final', cmap='YlOrRd', legend=True, ax=ax)
plt.title('Distribuição Regional das Notas de Matemática no Estado de SP')
plt.axis('off')
plt.show()

In [ ]:
# Criar o mapa de calor para idh_municipio
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
mapa_final.plot(column='idhm', cmap='YlGnBu', legend=True, ax=ax)
plt.title('Distribuição Regional do IDH dos Municípios de SP')
plt.axis('off')
plt.show()


# Conclusão

Considerações finais e sugestões de passos futuros para enriquecimento da análise
- evidências foram favoráveis ou contrárias à hipótese
- Sugerir aprofundamentos
